In [1]:
import os
import json
import csv

# --- CONFIGURATION ---
# IMPORTANT: Order matters! Patch files must go at the end so they overwrite the old "ERROR" records.
INPUT_FILES = [
    "/kaggle/input/datasets/mianafzalsaeed/nlp-mini-project-7b-4-bit-quantized-results/person_A_results.json",
    "/kaggle/input/datasets/mianafzalsaeed/nlp-mini-project-7b-4-bit-quantized-results/person_B_results.json",
    "/kaggle/input/datasets/mianafzalsaeed/nlp-mini-project-7b-4-bit-quantized-results/person_C_results.json",
    "/kaggle/input/datasets/mianafzalsaeed/nlp-mini-project-7b-4-bit-quantized-results/person_D_results.json",
    "/kaggle/input/datasets/mianafzalsaeed/nlp-mini-project-7b-4-bit-quantized-results/person_A_patch_600-799.json",
    "/kaggle/input/datasets/mianafzalsaeed/nlp-mini-project-7b-4-bit-quantized-results/person_A_patch_800-999.json"
]

MASTER_JSON_OUTPUT = "/kaggle/working/MASTER_7B_EXTREME_Results.json"
MASTER_CSV_OUTPUT = "/kaggle/working/MASTER_7B_EXTREME_Metrics.csv"

# MVBench Task Mapping by Index (200 questions per task)
MVBENCH_TASKS = [
    'action_sequence', 'moving_count', 'action_prediction', 'episodic_reasoning', 
    'action_antonym', 'action_count', 'scene_transition', 'object_shuffle', 
    'object_existence', 'fine_grained_pose', 'unexpected_action', 'moving_direction', 
    'state_change', 'object_interaction', 'character_order', 'action_localization', 
    'counterfactual_inference', 'fine_grained_action', 'moving_attribute', 'egocentric_navigation'
]

def get_task_name(index):
    task_idx = index // 200
    if 0 <= task_idx < len(MVBENCH_TASKS):
        return MVBENCH_TASKS[task_idx]
    return "unknown_task"

print("Commencing Grand Aggregation for 7B EXTREME COMPRESSION...\n")

# --- PHASE 1: MERGE AND OVERWRITE ---
master_records = {}

for file_path in INPUT_FILES:
    if os.path.exists(file_path):
        with open(file_path, "r") as f:
            try:
                data = json.load(f)
                loaded_count = 0
                patched_count = 0
                for record in data:
                    q_id = record["id"]
                    if q_id in master_records:
                        patched_count += 1
                    master_records[q_id] = record
                    loaded_count += 1
                
                print(f"[+] Loaded {loaded_count} records from {os.path.basename(file_path)} (Overwrote {patched_count} old records).")
            except json.JSONDecodeError:
                print(f"[!] ERROR: {os.path.basename(file_path)} is corrupted or empty.")
    else:
        print(f"[-] WARNING: {os.path.basename(file_path)} not found. Skipping.")

# Sort chronologically by ID
sorted_records = [master_records[k] for k in sorted(master_records.keys())]

# --- PHASE 2: CALCULATE METRICS ---
task_metrics = {task: {"correct": 0, "total": 0, "ttft_sum": 0.0, "tps_sum": 0.0} for task in MVBENCH_TASKS}
overall_correct = 0
overall_ttft = 0.0
overall_tps = 0.0
peak_vram = 0.0

for r in sorted_records:
    task_name = get_task_name(r["id"])
    is_correct = (r["response"].strip().upper() == r.get("ground_truth_letter", "").strip().upper())
    
    if is_correct:
        task_metrics[task_name]["correct"] += 1
        overall_correct += 1
        
    task_metrics[task_name]["total"] += 1
    task_metrics[task_name]["ttft_sum"] += r.get("ttft_seconds", 0.0)
    task_metrics[task_name]["tps_sum"] += r.get("tokens_per_second", 0.0)
    
    overall_ttft += r.get("ttft_seconds", 0.0)
    overall_tps += r.get("tokens_per_second", 0.0)
    
    if r.get("peak_vram_gb", 0.0) > peak_vram:
        peak_vram = r.get("peak_vram_gb", 0.0)

total_records = len(sorted_records)

# --- PHASE 3: CONSOLE REPORT ---
print("\n===========================================================================")
print(f"{'MVBench Sub-Task':<30} | {'Accuracy':<10} | {'Avg TTFT':<12} | {'Avg TPS'}")
print("===========================================================================")

csv_data = []
csv_headers = ["Task", "Accuracy_Percent", "Correct", "Total", "Avg_TTFT_sec", "Avg_TPS"]

for task in MVBENCH_TASKS:
    data = task_metrics[task]
    if data["total"] > 0:
        acc = (data["correct"] / data["total"]) * 100
        avg_ttft = data["ttft_sum"] / data["total"]
        avg_tps = data["tps_sum"] / data["total"]
        
        print(f"{task:<30} | {acc:>5.2f}%    | {avg_ttft:>6.2f} sec   | {avg_tps:>5.2f}")
        csv_data.append([task, round(acc, 2), data["correct"], data["total"], round(avg_ttft, 2), round(avg_tps, 2)])
    else:
        print(f"{task:<30} | NO DATA")

if total_records > 0:
    final_acc = (overall_correct / total_records) * 100
    final_avg_ttft = overall_ttft / total_records
    final_avg_tps = overall_tps / total_records

    print("===========================================================================")
    print(f"{'OVERALL ACCURACY':<30} | {final_acc:>5.2f}%    ({overall_correct}/{total_records})")
    print("===========================================================================\n")
    
    print("=============================================")
    print("       HARDWARE PERFORMANCE SUMMARY       ")
    print("=============================================")
    print(f"Average Time-to-First-Token:     {final_avg_ttft:.2f} seconds")
    print(f"Average Tokens-per-Second:       {final_avg_tps:.2f} tokens/s")
    print(f"Absolute Peak VRAM Usage:        {peak_vram:.2f} GB")
    print("=============================================\n")

# --- PHASE 4: DISK EXPORT ---
with open(MASTER_JSON_OUTPUT, "w") as f:
    json.dump(sorted_records, f, indent=4)

with open(MASTER_CSV_OUTPUT, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(csv_headers)
    writer.writerows(csv_data)
    
    if total_records > 0:
        writer.writerow([])
        writer.writerow(["OVERALL", round(final_acc, 2), overall_correct, total_records, round(final_avg_ttft, 2), round(final_avg_tps, 2)])
        writer.writerow(["PEAK_VRAM_GB", peak_vram, "", "", "", ""])

print(f"Aggregation Complete!")
print(f"Total Unique Records: {total_records} / 4000")
print(f"Master JSON saved to: {MASTER_JSON_OUTPUT}")
print(f"Metrics CSV saved to: {MASTER_CSV_OUTPUT}")

Commencing Grand Aggregation for 7B EXTREME COMPRESSION...

[+] Loaded 800 records from person_A_results.json (Overwrote 0 old records).
[+] Loaded 800 records from person_B_results.json (Overwrote 0 old records).
[+] Loaded 1000 records from person_C_results.json (Overwrote 0 old records).
[+] Loaded 1000 records from person_D_results.json (Overwrote 0 old records).
[+] Loaded 200 records from person_A_patch_600-799.json (Overwrote 0 old records).
[+] Loaded 200 records from person_A_patch_800-999.json (Overwrote 200 old records).

MVBench Sub-Task               | Accuracy   | Avg TTFT     | Avg TPS
action_sequence                | 39.00%    |   0.35 sec   | 54.35
moving_count                   | 41.50%    |   0.21 sec   | 39.34
action_prediction              | 34.00%    |   0.30 sec   | 51.99
episodic_reasoning             | 42.00%    |   0.40 sec   |  0.00
action_antonym                 | 80.50%    |   0.50 sec   |  0.00
action_count                   | 41.00%    |   0.35 sec   | 65

In [7]:
import json

# --- CONFIGURATION ---
# Change this to whatever your extreme compression JSON file for Person A was named
FILE_TO_AUDIT = "/kaggle/input/datasets/mianafzalsaeed/nlp-mini-project-7b-4-bit-quantized-results/person_A_patch_800-999.json" 

try:
    with open(FILE_TO_AUDIT, "r") as f:
        data = json.load(f)
        
    print("AUDIT: action_antonym (Extreme Compression)\n")
    print(f"{'ID':<5} | {'Model Answer':<15} | {'Ground Truth':<15}")
    print("-" * 40)
    
    count = 0
    for record in data:
        # Check if the ID falls into the action_antonym block
        if 800 <= record["id"] < 1000:
            resp = record.get("response", "N/A")
            gt = record.get("ground_truth_letter", "N/A")
            
            # Print the first 20 just to see what is happening
            print(f"{record['id']:<5} | {resp:<15} | {gt:<15}")
            
            count += 1
            if count >= 20: 
                break
                
except Exception as e:
    print(f"Error reading file: {e}")

AUDIT: action_antonym (Extreme Compression)

ID    | Model Answer    | Ground Truth   
----------------------------------------
800   | B               | C              
801   | A               | B              
802   | C               | C              
803   | B               | B              
804   | C               | C              
805   | C               | C              
806   | C               | C              
807   | C               | C              
808   | A               | A              
809   | A               | A              
810   | B               | B              
811   | B               | B              
812   | A               | A              
813   | A               | A              
814   | C               | C              
815   | C               | C              
816   | B               | A              
817   | A               | B              
818   | B               | B              
819   | C               | C              
